# 04 · Eleição do vencedor e promoção controlada

Regra (`config.yaml → promocao`), fixada antes dos ciclos:

1. **Vencedor do ciclo** = candidato com maior PR-AUC na validação.
2. **Checagens de qualidade**: ROC-AUC ≥ 0,75 e calibração (média prevista ÷ observada) entre 0,8 e 1,25.
3. **Sem modelo vigente** (primeiro ciclo): o vencedor é promovido.
4. **Com vigente**: o vigente é reavaliado **na mesma validação deste ciclo** — que ele não viu, porque foi treinado
   num ciclo anterior com corte mais antigo — e o vencedor só é promovido se superar o vigente em **≥ 0,002** de
   PR-AUC e passar nas checagens. Caso contrário, **o vigente é mantido**.

A decisão é registrada em `models/modelo_vigente.json` (com o histórico de todos os ciclos) e no MLflow.
Modelos substituídos vão para `models/historico/`.

In [ ]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [ ]:
import joblib
from src import modelagem as md, promocao as pr, preprocessamento as pp
ciclo = ler_ciclo(cfg)
validacao = pd.read_parquet(estado(cfg, "processados") / "validacao.parquet")
y_val = validacao[pp.ALVO].to_numpy()
candidatos = {a: c["metricas"] for a, c in ciclo["candidatos"].items()}

vigente, manifesto = pr.carregar_vigente(cfg)
if vigente is not None:
    v = manifesto["vigente"]
    m_vigente = md.metricas(y_val, md.prever(vigente, validacao, v["features"]))
    print(f"vigente: {v['algoritmo']} do ciclo {v['cycle_id']} ({v['data_version']})")
else:
    m_vigente = None
    print("sem modelo vigente: primeiro ciclo")
tabela = pd.DataFrame({**candidatos, **({"vigente": m_vigente} if m_vigente else {})}).T
tabela

In [ ]:
decisao = pr.decidir(candidatos, m_vigente, cfg)
print(f"DECISÃO: {decisao['decisao'].upper()} — {decisao['motivo']}")
decisao

In [ ]:
vencedor = ciclo["candidatos"][decisao["vencedor_ciclo"]]
modelo_vencedor = joblib.load(estado(cfg, "processados") / "candidatos" / f"{decisao['vencedor_ciclo']}.joblib")
manifesto = pr.registrar(cfg, ciclo, decisao, vencedor, modelo_vencedor)

mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · decisão: {decisao['decisao']}",
                      tags=tags_ciclo(ciclo, etapa="promocao", papel="decisao", decisao=decisao["decisao"])) as run:
    mlflow.log_params({"vencedor_ciclo": decisao["vencedor_ciclo"], "run_vencedor": vencedor["run_id"],
                       "decisao": decisao["decisao"], "motivo": decisao["motivo"][:500],
                       "vigente_apos": manifesto["vigente"]["cycle_id"], "limiar": cfg["promocao"]["limiar_melhoria"]})
    mlflow.log_metrics({"valor_vencedor": decisao["valor_vencedor"], "promovido": int(decisao["decisao"] == "promovido"),
                        **({"valor_vigente": decisao["valor_vigente"]} if decisao["valor_vigente"] is not None else {})})
    mlflow.log_artifact(pr.manifesto_path(cfg))
mlflow.MlflowClient().set_tag(vencedor["run_id"], "decisao", decisao["decisao"])

ev = pasta_evidencias(cfg, ciclo)
(ev / "decisao.json").write_text(json.dumps(decisao, indent=2, ensure_ascii=False))
tabela.to_csv(ev / "comparacao_com_vigente.csv")
ciclo = atualizar_ciclo(cfg, decisao=decisao["decisao"], run_decisao=run.info.run_id)
print(json.dumps(manifesto["vigente"], indent=2, ensure_ascii=False))